# Data Cleaning

**Client:** Bonnie Brown — a **seller** who owns a typical mid-tier house in one of King County's "middle-class" zip codes (zip codes whose median sale price falls in the 40th–60th percentile across the county — see the SQL in [`03_fetching_the_data_eda.ipynb`](03_fetching_the_data_eda.ipynb)). She wants to sell within the next **6–12 months** for the best possible price.

**Goal of this notebook:** take the raw pull from the database and turn it into a clean, analysis-ready dataset — correct data types, handle missing values and known data-entry errors, and document every decision — so the EDA in [`05_eda.ipynb`](05_eda.ipynb) rests on trustworthy data. As the saying goes: *garbage in, garbage out*.

**Scope:** every column from the raw pull is inspected and cleaned here (see [`column_names.md`](column_names.md) for the data dictionary). Deep, hypothesis-driven analysis of individual columns happens later in the EDA notebook — this notebook only makes sure the data itself is correct.

## Workflow
1. Initial inspection (shape, dtypes, missing values, duplicates)
2. Fix data types (dates, categoricals)
3. Fix known data-entry errors
4. Handle missing values (impute + flag, never silently drop)
5. Check duplicates / resales
6. Check internal consistency (sqft components) and outliers
7. Final sanity check and save the cleaned CSV

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df = pd.read_csv("data/bonnie_brown_middle_class_houses.csv")
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")
df.head()

Loaded 3,818 rows and 22 columns


,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price,sale_count
0,200500410,3.00,2.50,"1,960.00","9,535.00",2.00,0.00,0.00,3,8,"1,960.00",0.00,1989,NaN,98011,47.74,-122.22,"2,520.00","9,206.00",2015-05-06,"575,000.00",1
1,200500610,3.00,2.50,"2,600.00","7,465.00",2.00,0.00,0.00,3,9,"2,600.00",0.00,1988,NaN,98011,47.74,-122.22,"2,660.00","7,683.00",2014-06-11,"571,000.00",1
2,200500680,3.00,2.50,"2,620.00","11,056.00",2.00,0.00,0.00,3,9,"2,620.00",0.00,1988,0.00,98011,47.74,-122.22,"2,560.00","8,688.00",2014-07-15,"557,500.00",1
3,200500700,3.00,2.50,"2,120.00","9,736.00",2.00,0.00,0.00,3,9,"2,120.00",0.00,1988,NaN,98011,47.74,-122.22,"2,490.00","8,763.00",2014-06-12,"531,000.00",1
4,200510060,3.00,2.50,"2,570.00","9,487.00",2.00,0.00,3.00,3,9,"2,570.00",0.00,1989,0.00,98011,47.74,-122.22,"2,490.00","9,898.00",2014-12-31,"605,000.00",1


## 1. Initial Inspection

Before changing anything, we look at the raw shape of the problem: data types, how much is missing and where, and whether any rows are exact duplicates.

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             3818 non-null   int64  
 1   bedrooms       3818 non-null   float64
 2   bathrooms      3818 non-null   float64
 3   sqft_living    3818 non-null   float64
 4   sqft_lot       3818 non-null   float64
 5   floors         3818 non-null   float64
 6   waterfront     3390 non-null   float64
 7   view           3805 non-null   float64
 8   condition      3818 non-null   int64  
 9   grade          3818 non-null   int64  
 10  sqft_above     3818 non-null   float64
 11  sqft_basement  3744 non-null   float64
 12  yr_built       3818 non-null   int64  
 13  yr_renovated   3138 non-null   float64
 14  zipcode        3818 non-null   int64  
 15  lat            3818 non-null   float64
 16  long           3818 non-null   float64
 17  sqft_living15  3818 non-null   float64
 18  sqft_lot

In [3]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = (
    pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    .loc[missing > 0]
    .sort_values("missing_count", ascending=False)
)
missing_summary

,missing_count,missing_pct
yr_renovated,680,17.81
waterfront,428,11.21
sqft_basement,74,1.94
view,13,0.34


**Observations:**
- `waterfront` missing in ~11% of rows, `yr_renovated` in ~18%, `sqft_basement` in ~2%, `view` in <1%.
- No columns are missing *entirely* for any given row — this looks like sensor/reporting gaps per field, not corrupted records, so we impute rather than drop rows (dropping would throw away real houses from the exact market Bonnie cares about).

Next, check for exact duplicate rows and duplicate house `id`s.

In [4]:
print("Fully duplicated rows:", df.duplicated().sum())

dup_ids = df[df["id"].duplicated(keep=False)].sort_values(["id", "date"])
print("Rows sharing a house id:", len(dup_ids))
dup_ids[["id", "date", "price", "sale_count"]].head(10)

Fully duplicated rows: 0
Rows sharing a house id: 52


,id,date,price,sale_count
13,526059224,2014-09-23,"260,000.00",2
14,526059224,2015-02-06,"470,000.00",2
2422,722039087,2014-09-23,"220,500.00",2
2423,722039087,2015-05-04,"329,000.00",2
3537,1250201165,2014-11-21,"441,000.00",2
3536,1250201165,2015-03-17,"474,500.00",2
527,1524079093,2014-08-27,"275,000.00",2
528,1524079093,2015-03-18,"369,500.00",2
962,1974300020,2014-08-27,"380,000.00",2
961,1974300020,2015-02-18,"624,900.00",2


**Observation:** There are no fully-duplicated rows. 26 house `id`s appear twice, each with a *different* `date` and `price` — these are genuine **resales**: the same physical house sold twice within our data window (2014–2015). This is real signal, not a data error, so we **keep every row** (each row is one sale event) and simply note it via the existing `sale_count` column. This is actually a nice bonus for Bonnie: resold houses hint at short-term price appreciation in her area.

## 2. Fix Data Types

- `date` is currently a string → convert to `datetime`, and derive `sale_year`, `sale_month`, and `sale_season` so we can later test whether **timing** affects sale price (directly relevant to Bonnie's "when should I list?" question).
- Several numerically-coded columns are really **categories**, not continuous numbers: `waterfront` (yes/no), `view` (quality tier 0–4), `condition` (1–5), `grade` (4–13), `zipcode`. We convert these to pandas `category` dtype so they plot and group correctly later.
- `sqft_*` columns are stored as floats but are always whole numbers → cast to `int`.

In [5]:
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")

df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month

season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall",
}
df["sale_season"] = df["sale_month"].map(season_map)

df[["date", "sale_year", "sale_month", "sale_season"]].head()

,date,sale_year,sale_month,sale_season
0,2015-05-06,2015,5,Spring
1,2014-06-11,2014,6,Summer
2,2014-07-15,2014,7,Summer
3,2014-06-12,2014,6,Summer
4,2014-12-31,2014,12,Winter


In [6]:
# Whole-number sqft columns that have no missing values -> safe to cast now.
# sqft_basement, waterfront, view are cast later, after their missing values are imputed.
whole_number_cols = ["bedrooms", "sqft_living", "sqft_lot", "sqft_above", "sqft_living15", "sqft_lot15"]
df[whole_number_cols] = df[whole_number_cols].astype(int)

# Categoricals with no missing values -> convert now.
df["condition"] = df["condition"].astype("category")
df["grade"] = df["grade"].astype("category")
df["zipcode"] = df["zipcode"].astype("category")

df.dtypes

id                        int64
bedrooms                  int64
bathrooms               float64
sqft_living               int64
sqft_lot                  int64
floors                  float64
waterfront              float64
view                    float64
condition              category
grade                  category
sqft_above                int64
sqft_basement           float64
yr_built                  int64
yr_renovated            float64
zipcode                category
lat                     float64
long                    float64
sqft_living15             int64
sqft_lot15                int64
date             datetime64[ns]
price                   float64
sale_count                int64
sale_year                 int32
sale_month                int32
sale_season              object
dtype: object

## 3. Fix a Known Data-Entry Error: `yr_renovated`

The max `yr_renovated` in the raw data is `20140`, and non-zero values like `19890`, `20050`, `19440` are all impossible years. There's an **extra trailing zero** on every recorded renovation year — almost certainly a form/export bug (`2014` stored as `20140`). We can detect and fix this safely: any `yr_renovated` greater than the most recent sale year (2015) is off by a factor of 10. `0` (never renovated) is unaffected and left as-is.

In [9]:
bad_renovation_years = df["yr_renovated"] > df["sale_year"].max()
print(f"Rows with an impossible yr_renovated (extra trailing zero): {bad_renovation_years.sum()}")
print(sorted(df.loc[bad_renovation_years, "yr_renovated"].unique()))

df.loc[bad_renovation_years, "yr_renovated"] = df.loc[bad_renovation_years, "yr_renovated"] / 10

# sanity check: no renovation year should be before the house was built, or after it was sold
# (NaNs are excluded here -- they still represent "not yet recorded" and get imputed in step 4)
renovated = df["yr_renovated"].dropna()
assert (renovated <= df.loc[renovated.index, "sale_year"]).all()
assert ((renovated == 0) | (renovated >= df.loc[renovated.index, "yr_built"])).all()
print("yr_renovated is now within a same range:", df["yr_renovated"].max())

Rows with an impossible yr_renovated (extra trailing zero): 0
[]
yr_renovated is now within a same range: 2014.0


In [13]:
df.head()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price,sale_count,sale_year,sale_month,sale_season,sqft_basement_was_missing,waterfront_was_missing,view_was_missing,yr_renovated_was_missing,is_renovated
0,200500410,3,2.50,1960,9535,2.00,0.00,0.00,3,8,1960,0,1989,0.00,98011,47.74,-122.22,2520,9206,2015-05-06,"575,000.00",1,2015,5,Spring,False,False,False,True,False
1,200500610,3,2.50,2600,7465,2.00,0.00,0.00,3,9,2600,0,1988,0.00,98011,47.74,-122.22,2660,7683,2014-06-11,"571,000.00",1,2014,6,Summer,False,False,False,True,False
2,200500680,3,2.50,2620,11056,2.00,0.00,0.00,3,9,2620,0,1988,0.00,98011,47.74,-122.22,2560,8688,2014-07-15,"557,500.00",1,2014,7,Summer,False,False,False,False,False
3,200500700,3,2.50,2120,9736,2.00,0.00,0.00,3,9,2120,0,1988,0.00,98011,47.74,-122.22,2490,8763,2014-06-12,"531,000.00",1,2014,6,Summer,False,False,False,True,False
4,200510060,3,2.50,2570,9487,2.00,0.00,3.00,3,9,2570,0,1989,0.00,98011,47.74,-122.22,2490,9898,2014-12-31,"605,000.00",1,2014,12,Winter,False,False,False,False,False


## 4. Handle Missing Values

Best practice: never silently drop or fill data without recording that it happened. For each column with missing values we:
1. Add a boolean `<column>_was_missing` flag **before** imputing, so we can always check later whether "no renovation" and "renovation unknown" behave differently.
2. Impute with the value that matches this dataset's convention:

| Column | Missing meaning | Imputed with |
| --- | --- | --- |
| `waterfront` | Not recorded → assume not waterfront (rare, ~1% of known values are `1`) | `0` (No) |
| `view` | Not recorded → assume no special view | `0` (mode, and the "no view" baseline) |
| `sqft_basement` | Not recorded → **derive** it instead of guessing, since `sqft_living` and `sqft_above` are never missing | `sqft_living - sqft_above` |
| `yr_renovated` | Not recorded → assume never renovated | `0` |

> Deriving `sqft_basement` this way is strictly more accurate than assuming "no basement," and it also fixes the 28 rows found earlier where `sqft_above + sqft_basement != sqft_living`.

In [11]:
df["sqft_basement_was_missing"] = df["sqft_basement"].isna()
df["sqft_basement"] = df["sqft_basement"].fillna(df["sqft_living"] - df["sqft_above"])
df["sqft_basement"] = df["sqft_basement"].astype(int)

for col in ["waterfront", "view", "yr_renovated"]:
    df[f"{col}_was_missing"] = df[col].isna()
    df[col] = df[col].fillna(0)

df["waterfront"] = df["waterfront"].astype("category")
df["view"] = df["view"].astype("category")

# derived convenience flag used throughout the EDA
df["is_renovated"] = df["yr_renovated"] > 0

print("Remaining missing values:", df.isna().sum().sum())
df[[c for c in df.columns if c.endswith("_was_missing")]].sum()

Remaining missing values: 0


sqft_basement_was_missing     74
waterfront_was_missing       428
view_was_missing              13
yr_renovated_was_missing     680
dtype: int64

## 5. Internal Consistency Check

`sqft_above` + `sqft_basement` should always equal `sqft_living`. Since we derived the missing `sqft_basement` values from this exact identity, this should now hold for every row.

In [12]:
sqft_mismatch = df["sqft_above"] + df["sqft_basement"] != df["sqft_living"]
print(f"Rows where sqft_above + sqft_basement != sqft_living: {sqft_mismatch.sum()}")
assert sqft_mismatch.sum() == 0, "Unexpected inconsistency remains"

Rows where sqft_above + sqft_basement != sqft_living: 0


## 6. Outlier Review

We check `price` for statistical outliers using the IQR rule, purely to **understand** the shape of the market — not to blindly delete anything. For a seller like Bonnie, an unusually high sale in her zip code isn't noise to discard, it's exactly the kind of ceiling-setting comparable she'd want to know about. Deep distribution/skew analysis happens in the EDA notebook; here we just confirm there's nothing that looks like a data error (e.g. a $0 sale, a typo with too many zeros).

In [14]:
q1, q3 = df["price"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
price_outliers = (df["price"] < lower_bound) | (df["price"] > upper_bound)

print(f"Price IQR bounds: ${lower_bound:,.0f} - ${upper_bound:,.0f}")
print(f"Outlier sales by IQR rule: {price_outliers.sum()} ({price_outliers.mean():.1%})")
print(f"Min price: ${df['price'].min():,.0f} | Max price: ${df['price'].max():,.0f}")

Price IQR bounds: $80,016 - $855,991
Outlier sales by IQR rule: 240 (6.3%)
Min price: $80,000 | Max price: $3,600,000


**Decision:** ~6% of sales sit outside the IQR whiskers, all on the high side, and the min/max are both plausible real house prices (no $0 or 12-zero typos). We **keep all rows** — these are legitimate high-end sales within otherwise middle-class zip codes, which is directly useful evidence for Bonnie (they show the upside ceiling in her market). We do flag this for the EDA notebook, where we'll likely use a log scale or median-based stats so these don't distort the visuals.

## 7. Final Sanity Check and Save

One last look at the cleaned dataset before saving it, so `05_eda.ipynb` can load a single, ready-to-use, well-documented file.

In [17]:
print("Shape:", df.shape)
print("\nMissing values remaining:", df.isna().sum().sum())
print("\nDtypes:")
print(df.dtypes)
df.head()

Shape: (3818, 30)

Missing values remaining: 0

Dtypes:
id                                    int64
bedrooms                              int64
bathrooms                           float64
sqft_living                           int64
sqft_lot                              int64
floors                              float64
waterfront                         category
view                               category
condition                          category
grade                              category
sqft_above                            int64
sqft_basement                         int64
yr_built                              int64
yr_renovated                        float64
zipcode                            category
lat                                 float64
long                                float64
sqft_living15                         int64
sqft_lot15                            int64
date                         datetime64[ns]
price                               float64
sale_count          

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price,sale_count,sale_year,sale_month,sale_season,sqft_basement_was_missing,waterfront_was_missing,view_was_missing,yr_renovated_was_missing,is_renovated
0,200500410,3,2.50,1960,9535,2.00,0.00,0.00,3,8,1960,0,1989,0.00,98011,47.74,-122.22,2520,9206,2015-05-06,"575,000.00",1,2015,5,Spring,False,False,False,True,False
1,200500610,3,2.50,2600,7465,2.00,0.00,0.00,3,9,2600,0,1988,0.00,98011,47.74,-122.22,2660,7683,2014-06-11,"571,000.00",1,2014,6,Summer,False,False,False,True,False
2,200500680,3,2.50,2620,11056,2.00,0.00,0.00,3,9,2620,0,1988,0.00,98011,47.74,-122.22,2560,8688,2014-07-15,"557,500.00",1,2014,7,Summer,False,False,False,False,False
3,200500700,3,2.50,2120,9736,2.00,0.00,0.00,3,9,2120,0,1988,0.00,98011,47.74,-122.22,2490,8763,2014-06-12,"531,000.00",1,2014,6,Summer,False,False,False,True,False
4,200510060,3,2.50,2570,9487,2.00,0.00,3.00,3,9,2570,0,1989,0.00,98011,47.74,-122.22,2490,9898,2014-12-31,"605,000.00",1,2014,12,Winter,False,False,False,False,False


In [19]:
df.to_csv("data/bonnie_brown_middle_class_houses_clean.csv", index=False)
print("Saved cleaned dataset to data/bonnie_brown_middle_class_houses_clean.csv")

Saved cleaned dataset to data/bonnie_brown_middle_class_houses_clean.csv


# Appendix: Automated Profiling Report

As an extra sanity check (not a replacement for the deliberate cleaning above), we run [`ydata-profiling`](https://github.com/ydataai/ydata-profiling) — an industry-standard tool that auto-generates warnings for skew, high correlation, cardinality issues, etc. This is a good second pair of eyes to catch anything the manual pass above missed, run on the **cleaned** data.

In [ ]:
from ydata_profiling import ProfileReport

df_clean = pd.read_csv("data/bonnie_brown_middle_class_houses_clean.csv")

profile = ProfileReport(
    df_clean, title="Bonnie Brown - King County Housing EDA (Cleaned Data)", explorative=True
)
profile.to_notebook_iframe()

# Save Report to Standalone file

In [ ]:
profile.to_file("reports/bonnie_brown_eda_report.html")